In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [7]:
# 모든 라이브러리를 설치 후에 세션을 다시 시작해주세요
# Colab 셀에 한 번만 실행
!pip install -q protobuf==3.20.3 gensim==4.3.1
!pip -q uninstall -y scipy
!pip -q install scipy==1.9.3
!pip -q install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 85.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.12.0 requires protobuf<6.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.4/33.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 107.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages tha

# 라이브러리

In [2]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 시각화
import matplotlib.pyplot as plt

# Tensorflow 라이브러리
import tensorflow as tf

# Tensorflow 전처리
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# glove 모델
import gensim.downloader as api

# Tensorflow 모델 구조
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Conv1D, GlobalMaxPooling1D, Embedding, Conv2D, Dense, Input, Flatten,
    Concatenate, Attention, Multiply, Add, Reshape, MaxPool2D
)
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.utils import plot_model

# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error


# Utils


In [3]:
# 텍스트 패딩을 생성한 후, 리스트의 형태로 DataFrame에 저장
def create_pad_sequence(df, col, max_sequence_length = 10):

    # 토큰화 및 패딩
    text_sequence = tokenizer.texts_to_sequences(df[col])
    padding_result = pad_sequences(text_sequence, maxlen=max_sequence_length, padding='post')
    padding_result # 새로 만들어준 사전에서의 인덱스

    # 'padded_sequences' 열을 df에 추가하고, 패딩이 적용된 시퀀스의 내용을 할당
    df[col + '_padded_sequences'] = list(padding_result)

    # 'padded_sequences' 열의 데이터 타입을 object로 변경하여 각 요소를 리스트 형태로 저장
    df[col + '_padded_sequences'] = df[col + '_padded_sequences'].apply(lambda x: list(x))

# 데이터셋 로딩

In [4]:
# 데이터 로딩
df = pd.read_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_v4.json')

print(df.shape)
df.head()

(23491, 15)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb
0,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BJKYJR3L,B0BJKYJR3L,"Nice privacy shield. Good color, fits well, et...",3,Jephuwneh 2 Pack Gradient Tempered Glass for i...,"[-0.0753104165, 0.1205661818, -0.0582872108, -...",https://m.media-amazon.com/images/I/515oPLx13k...,https://m.media-amazon.com/images/I/515oPLx13k...,https://m.media-amazon.com/images/I/515oPLx13k...,"[9.4065265656, 28.670337677, 22.95885849, 26.4...","[-0.5072865486, -0.22111754120000002, 0.448988...","[-0.35559991, -0.19625245030000002, 0.33806100...","[-0.0538860857, 0.1409395933, -0.0256068446, -...","[-0.0548222102, 0.1186653376, -0.0376601368, -..."
1,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BHTZQHQW,B0BHTZQHQW,"This thing is great. It's not too big, has a d...",5,"NEOCOR Magnetic Wireless Power Bank, 10,000mAh...","[-0.1076164767, 0.15895271300000002, -0.038697...",https://m.media-amazon.com/images/I/71K8BDxzsA...,https://m.media-amazon.com/images/I/71K8BDxzsA...,https://m.media-amazon.com/images/I/71K8BDxzsA...,"[15.0432891846, 40.8174591064, 0.0, 27.5127182...","[-0.5072865486, -0.22111754120000002, 0.448988...","[-0.5915018916, -0.6934525371, 0.2179201096, -...","[-0.0538860857, 0.1409395933, -0.0256068446, -...","[-0.0799105316, 0.16881611940000002, -0.014415..."
2,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B9N4VRB9,B0BHSSCXB2,This is a much better way to mount my iPhone (...,5,Lamicall Gooseneck Phone Holder Bed - [Newest ...,"[-0.098338306, 0.1566942036, -0.0378348567, -0...",https://m.media-amazon.com/images/I/71b+Sx-Jq7...,https://m.media-amazon.com/images/I/71b+Sx-Jq7...,https://m.media-amazon.com/images/I/71b+Sx-Jq7...,"[0.0, 57.4749221802, 14.7353601456, 36.8316497...","[-0.5072865486, -0.22111754120000002, 0.448988...","[-0.1125981882, -0.2578130662, 0.468741864, -0...","[-0.0538860857, 0.1409395933, -0.0256068446, -...","[-0.08550157400000001, 0.1129528657, -0.022212..."
3,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B93H178F,B0B93H178F,"This is a fantastic charger, especially if you...",5,"Shargeek USB C Charger 140W Wall Charger, Mult...","[-0.0699621961, 0.1709383726, -0.0337884836, -...",https://m.media-amazon.com/images/I/61R6UXrAKT...,https://m.media-amazon.com/images/I/61R6UXrAKT...,https://m.media-amazon.com/images/I/61R6UXrAKT...,"[21.9141178131, 16.1274204254, 0.0, 10.3457145...","[-0.5072865486, -0.22111754120000002, 0.448988...","[-0.6311548352, -0.1796810478, 0.4211614132, -...","[-0.0538860857, 0.1409395933, -0.0256068446, -...","[-0.053146679, 0.1396851987, -0.0163876414, -0..."
4,AFERCDY2EFJKT7QUQ75GISNHTFOQ,B0BPXZWCBG,B0BPY47SY6,There never seems to be enough charging cords ...,4,"Charging Station for Multiple Devices,60W 12A ...","[-0.09403285380000001, 0.16169938450000002, -0...",https://m.media-amazon.com/images/I/51fqzioGKe...,https://m.media-amazon.com/images/I/51fqzioGKe...,https://m.media-amazon.com/images/I/51fqzioGKe...,"[0.0, 25.8072109222, 0.0, 35.9537582397, 0.0, ...","[-0.3012515306, -0.1072597727, 0.3101685643, -...","[-0.4422509968, -0.18747162820000002, 0.236477...","[-0.0667454079, 0.15736866, -0.0147655336, -0....","[-0.08094199, 0.1242809743, -0.0082041509, -0...."


# Text Embedding

In [5]:
# 최대 사용할 단어 수와 시퀀스의 최대 길이
max_words = 100000

# 토크나이저 초기화 및 학습
tokenizer = Tokenizer(num_words=max_words)

In [6]:
# lemmatized text 기준으로 사전 만들어주기
keyword = df['text'].explode().unique()
tokenizer.fit_on_texts(keyword)

# Create Embedding Matrix

In [7]:
# # glove 모델 다운로드
glove_model = api.load("glove-wiki-gigaword-100")

# Print the shape of the model to confirm its size
print('Model shape:', glove_model.vectors.shape)

Model shape: (400000, 100)


In [8]:
# 최대 사용할 단어 수와 시퀀스의 최대 길이
max_words = 100000

# 토크나이저 초기화 및 학습
tokenizer = Tokenizer(num_words=max_words)

# lemmatized text 기준으로 사전 만들어주기
keyword = df['text'].explode().unique()
tokenizer.fit_on_texts(keyword)

In [9]:
# 단어 인덱스 맵 생성 및 임베딩 행렬 준비
# 임베딩 차원을 100으로 가정
word_index = tokenizer.word_index
total_words = min(max_words, len(word_index))
embedding_matrix = np.zeros((total_words + 1, 100))

# 새로 만든 사전에 맞게 임베딩 해줘야함
# 임베딩 행렬의 크기 출력
for word, i in word_index.items():
    if i > max_words:
        continue
    if word in glove_model:
        embedding_matrix[i] = glove_model[word]

print(embedding_matrix)

[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [-0.038194   -0.24487001  0.72812003 ... -0.1459      0.82779998
   0.27061999]
 [-0.30664     0.16821     0.98510998 ... -0.38775     0.36916
   0.54521   ]
 ...
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]


In [10]:
# 최종 단어 index의 길이가 max word를 넘는 상황, 65만개 dataset 기준
len(tokenizer.word_index)

27784

# User, Item Embedding을 위한 전처리
- user별, item별 임베딩을 생성한 다음, 전체 DataFrame에 concat할 예정
- 실행 시간 단축을 위한 방안

In [11]:
df['user_id'].nunique(), df['asin'].nunique()

(4149, 4975)

In [12]:
# user별, item별로 텍스트를 연결
user_text_df = df.groupby('user_id')['text'].apply(lambda x: ' '.join(x)).reset_index()
item_text_df = df.groupby('asin')['text'].apply(lambda x: ' '.join(x)).reset_index()

# column 변경
user_text_df.columns = ['user_id', 'user_text']
item_text_df.columns = ['asin', 'item_text']

# 결과 확인
print(user_text_df.shape)
print(item_text_df.shape)

user_text_df.head()

(4149, 2)
(4975, 2)


,user_id,user_text
0,AE22Z3RLVIRU6RT5PNRK5CFFNEFQ,I bought the mirror version and love it. It lo...
1,AE22ZYH6WIY6WJHJRWOMPQCDJHHA,Extra Cables! ‘Nuff Said. I needed a replaceme...
2,AE23CDET66BP6FPOLJ6W4A7W5UWA,This case is pretty good: it has a rubbery fee...
3,AE23Y57MOO55MNGPWP3K3HI4SV7Q,It’s magnetic and piped right in the back of m...
4,AE23ZBUF2YVBQPH2NN6F5XSA3QYQ,I do like my regular Fitbit band for everyday ...


In [13]:
# sequence 길이 500으로 embedding 생성
create_pad_sequence(user_text_df,'user_text', max_sequence_length = 256)
create_pad_sequence(item_text_df,'item_text', max_sequence_length = 256)

# Data Concat

In [14]:
# 원래의 데이터셋에 결합해서 최종 데이터셋을 만들어줌
df = pd.merge(df, user_text_df, on='user_id', how='left')
df = pd.merge(df, item_text_df, on='asin', how='left')

# User, Item label encoding
- 라벨 인코딩
- train, test 분리



In [15]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

In [16]:
# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

Num_User = n_users + 1
Num_Item = n_items + 1

# train과 test data를 분리
train, test = train_test_split(df, shuffle=True, test_size=0.2, random_state=42)

In [17]:
# inputs
user_text_train = np.array(train['user_text_padded_sequences'].tolist(),dtype='int64')
user_text_test = np.array(test['user_text_padded_sequences'].tolist(),dtype='int64')

item_text_train = np.array(train['item_text_padded_sequences'].tolist(),dtype='int64')
item_text_test = np.array(test['item_text_padded_sequences'].tolist(),dtype='int64')

user_id_train = np.array(train['user_id'])
item_id_train = np.array(train['asin'])
user_id_test = np.array(test['user_id'])
item_id_test = np.array(test['asin'])

# rating
y_train = train['rating'].to_numpy()
y_test = test['rating'].to_numpy()

# AENAR

In [18]:
class emb_CNN(tf.keras.layers.Layer):
    def __init__(self, n_kernel):
        super(emb_CNN, self).__init__()
        self.conv1 = tf.keras.layers.Conv2D(filters=1, kernel_size=(2,100), strides = 2, activation='relu')
        self.flatten = tf.keras.layers.Flatten()
        self.dense = tf.keras.layers.Dense(units=n_kernel, activation='relu')

    def call(self, input):
        x = self.conv2(input)
        flat_cnn = self.flatten(x)
        output = self.dense(flat_cnn)
        return output

In [19]:
def aenar(user_num, item_num, id_dims, glove_dim, learning_rate=0.0001):

    # User
    user_input = Input(shape=(1,), dtype='int32', name='UserInput')
    user_embedding = Embedding(user_num, id_dims, name='UserIDEmb')(user_input)
    user_embedding = Flatten(name='UserFlatten')(user_embedding)

    # item
    item_input = Input(shape=(1,), dtype='int32', name='ItemInput')
    item_embedding = Embedding(item_num, id_dims, name='ItemIDEmb')(item_input)
    item_embedding = Flatten(name='ItemFlatten')(item_embedding)

    # Review
    user_review = Input(shape=(256,), name='CNN_user_Input')
    user_review_embedding_layer = Embedding(input_dim=len(embedding_matrix), output_dim=100, weights=[embedding_matrix], input_length=256, trainable=False)(user_review)
    user_conv_input = Reshape((256, 100, 1))(user_review_embedding_layer)

    item_review = Input(shape=(256,), name='CNN_item_Input')
    item_review_embedding_layer = Embedding(input_dim=len(embedding_matrix), output_dim=100, weights=[embedding_matrix], input_length=256, trainable=False)(item_review)
    item_conv_input = Reshape((256, 100, 1))(item_review_embedding_layer)

    # CNN
    user_review_cnn = Conv2D(filters=100, kernel_size=(2, 100), strides=2, activation='relu')(user_conv_input)
    user_review_pool = MaxPool2D(pool_size=(2, 1), strides=(2, 1))(user_review_cnn)  # MaxPooling layer added
    user_review_dense = Flatten()(user_review_pool)
    user_review_dense = Dense(id_dims, activation='relu')(user_review_dense)

    item_review_cnn = Conv2D(filters=100, kernel_size=(2, 100), strides=2, activation='relu')(item_conv_input)
    item_review_pool = MaxPool2D(pool_size=(2, 1), strides=(2, 1))(item_review_cnn)
    item_review_dense = Flatten()(item_review_pool)
    item_review_dense = Dense(id_dims, activation='relu')(item_review_dense)

    # Feature Fusion
    user_rep = Add()([user_embedding, user_review_dense])
    user_rep = Dense(id_dims, activation="relu")(user_rep)

    item_rep = Add()([item_embedding, item_review_dense])
    item_rep = Dense(id_dims, activation="relu")(item_rep)

    # Attentive Interaction
    attentive_input = Multiply()([user_rep, item_rep])
    attentive_rep = Dense(id_dims, activation="softmax")(attentive_input)

    final_rep = Multiply()([user_rep, item_rep, attentive_rep])

    # Rating Prediction
    dense = Dense(128, activation='relu')(final_rep)
    dense = Dense(64, activation='relu')(dense)
    output = Dense(1, activation='linear')(dense)

    # Model definition
    model = Model(inputs=[user_input, item_input, user_review, item_review], outputs=output)

    # 모델 컴파일
    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = learning_rate), loss = "mse", metrics = ["mse", "mae"])

    return model

In [20]:
# 모델 생성
model = aenar(user_num=Num_User, item_num=Num_Item, id_dims=128, glove_dim=100)
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ CNN_user_Input      │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CNN_item_Input      │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 256, 100)  │  2,778,500 │ CNN_user_Input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 256, 100)  │  2,778,500 │ CNN_item_Input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 256, 100,  │          0 │ embedding[0][0]   │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 256, 100,  │          0 │ embedding_1[0][0] │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 1,    │     20,100 │ reshape[0][0]     │
│                     │ 100)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 1,    │     20,100 │ reshape_1[0][0]   │
│                     │ 100)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ UserInput           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 1,     │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 100)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ItemInput           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 1,     │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 100)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ UserIDEmb           │ (None, 1, 128)    │    531,200 │ UserInput[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 6400)      │          0 │ max_pooling2d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ItemIDEmb           │ (None, 1, 128)    │    636,928 │ ItemInput[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 6400)      │          0 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ UserFlatten         │ (None, 128)       │          0 │ UserIDEmb[0][0]   │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 8,478,353 (32.34 MB)

 Trainable params: 2,921,353 (11.14 MB)

 Non-trainable params: 5,557,000 (21.20 MB)

In [21]:
# Configuration of EarlyStopping
early_stopping = EarlyStopping(
    monitor ='val_loss',  # Monitor model's validation loss
    min_delta = 0.001,     # Minimum change to qualify as an improvement
    patience = 5,         # Stop after 5 epochs without improvement
    verbose = 1,           # Print messages
    mode = 'min',          # Stop training when the quantity monitored has stopped decreasing
    restore_best_weights = True  # Restore model weights from the epoch with the best value of the monitored quantity
)

In [22]:
# 모델 학습
history = model.fit(
    x=[user_id_train, item_id_train, user_text_train, item_text_train],
    y=y_train,
    validation_split=0.125,
    batch_size= 128,
    callbacks=[early_stopping],
    epochs=50
)

Epoch 1/50
129/129 ━━━━━━━━━━━━━━━━━━━━ 16s 46ms/step - loss: 14.4806 - mae: 3.3231 - mse: 14.4806 - val_loss: 0.9802 - val_mae: 0.6807 - val_mse: 0.9802
Epoch 2/50
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.8149 - mae: 0.6269 - mse: 0.8149 - val_loss: 0.6847 - val_mae: 0.5656 - val_mse: 0.6847
Epoch 3/50
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6445 - mae: 0.5653 - mse: 0.6445 - val_loss: 0.6160 - val_mae: 0.5593 - val_mse: 0.6160
Epoch 4/50
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5605 - mae: 0.5300 - mse: 0.5605 - val_loss: 0.5879 - val_mae: 0.5534 - val_mse: 0.5879
Epoch 5/50
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5344 - mae: 0.5140 - mse: 0.5344 - val_loss: 0.5717 - val_mae: 0.5691 - val_mse: 0.5717
Epoch 6/50
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4958 - mae: 0.4878 - mse: 0.4958 - val_loss: 0.5741 - val_mae: 0.5870 - val_mse: 0.5741
Epoch 7/50
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4876 - mae: 0.4797 - mse: 0.4876 -

In [24]:
# 모델 성능 측정
predicted_ratings = model.predict([user_id_test, item_id_test, user_text_test, item_text_test])

# MAE
MAE = mean_absolute_error(y_test, predicted_ratings)
print(f"MAE: {MAE:.4f}")

# MSE
MSE = mean_squared_error(y_test, predicted_ratings)
print(f"MSE: {MSE:.4f}")

# RMSE
RMSE = np.sqrt(MSE)
print(f"RMSE: {RMSE:.4f}")

# MAPE
MAPE = mean_absolute_percentage_error(y_test, predicted_ratings) * 100
print(f"MAPE: {MAPE:.4f}%")

147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
MAE: 0.5154
MSE: 0.5396
RMSE: 0.7346
MAPE: 16.0519%


In [27]:
data = {
    'NeuMF': {
        'MAE': [0.602, 0.596, 0.592, 0.589, 0.590],
        'MSE': [0.589, 0.587, 0.589, 0.584, 0.585],
        'RMSE': [0.768, 0.766, 0.767, 0.764, 0.765],
        'MAPE': [0.182, 0.182, 0.181, 0.181, 0.181]
    },
    'MFNR': {
        'MAE': [0.509, 0.526, 0.510, 0.505, 0.515],
        'MSE': [0.508, 0.515, 0.505, 0.509, 0.506],
        'RMSE': [0.713, 0.718, 0.710, 0.713, 0.711],
        'MAPE': [0.159, 0.161, 0.158, 0.158, 0.159]
    },
    'ANR': {
        'MAE': [0.555, 0.547, 0.534, 0.539, 0.529],
        'MSE': [0.570, 0.575, 0.570, 0.571, 0.580],
        'RMSE': [0.755, 0.758, 0.755, 0.755, 0.761],
        'MAPE': [0.169, 0.168, 0.167, 0.168, 0.167]
    },
    'AFRAM': {
        'MAE': [0.515, 0.529, 0.488, 0.524, 0.502],
        'MSE': [0.523, 0.529, 0.525, 0.531, 0.531],
        'RMSE': [0.723, 0.728, 0.725, 0.729, 0.729],
        'MAPE': [0.162, 0.164, 0.157, 0.162, 0.161]
    },
    'AENAR': {
        'MAE': [0.509, 0.491, 0.520, 0.499, 0.515],
        'MSE': [0.545, 0.549, 0.555, 0.542, 0.540],
        'RMSE': [0.739, 0.741, 0.745, 0.736, 0.735],
        'MAPE': [0.162, 0.159, 0.162, 0.159, 0.161]
    },
    'CAMRec': {
        'MAE': [0.452, 0.439, 0.431, 0.455, 0.446],
        'MSE': [0.396, 0.406, 0.426, 0.399, 0.408],
        'RMSE': [0.629, 0.637, 0.653, 0.632, 0.639],
        'MAPE': [0.133, 0.139, 0.141, 0.133, 0.138]
    },
}

# 결과 저장용 DataFrame 생성
results = []

for algo, metrics in data.items():
    for metric, values in metrics.items():
        mean = np.mean(values)
        std = np.std(values, ddof=1)  # 표본 표준편차
        results.append([algo, metric, mean, std])

df = pd.DataFrame(results, columns=["Algorithm", "Metric", "Mean", "Std"])

# 보기 쉽게 피벗 테이블로 변환
pivot_df = df.pivot(index="Algorithm", columns="Metric", values=["Mean", "Std"])

print(pivot_df.round(4))

             Mean                             Std                        
Metric        MAE    MAPE     MSE    RMSE     MAE    MAPE     MSE    RMSE
Algorithm                                                                
AENAR      0.5068  0.1606  0.5462  0.7392  0.0118  0.0015  0.0060  0.0040
AFRAM      0.5116  0.1612  0.5278  0.7268  0.0167  0.0026  0.0036  0.0027
ANR        0.5408  0.1678  0.5732  0.7568  0.0104  0.0008  0.0043  0.0027
CAMRec     0.4446  0.1368  0.4070  0.6380  0.0098  0.0036  0.0117  0.0093
MFNR       0.5130  0.1590  0.5086  0.7130  0.0081  0.0012  0.0039  0.0031
NeuMF      0.5938  0.1814  0.5868  0.7660  0.0053  0.0005  0.0023  0.0016
